# Exploratory Data Analysis — Amazon ESCI Dataset
**Query-Product Relevance Classification**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Load data
examples = pd.read_parquet('../data/shopping_queries_dataset_examples.parquet')
products = pd.read_parquet('../data/shopping_queries_dataset_products.parquet')

df = examples.merge(products[['product_id', 'product_title']], on='product_id', how='left')
df = df[df['product_locale'] == 'us'].reset_index(drop=True)
print(f'Total samples: {len(df)}')
df.head()

## 1. Class Distribution

In [ ]:
label_map = {'E': 'Exact', 'S': 'Substitute', 'C': 'Complement', 'I': 'Irrelevant'}
df['label_name'] = df['esci_label'].map(label_map)

counts = df['label_name'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(counts.index, counts.values, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
axes[0].set_title('Class Distribution (Count)', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, str(v), ha='center')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#4C72B0','#DD8452','#55A868','#C44E52'])
axes[1].set_title('Class Distribution (%)', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/eda_class_distribution.png', bbox_inches='tight')
plt.show()

## 2. Query Length Distribution

In [ ]:
df['query_len'] = df['query'].str.split().str.len()
df['title_len'] = df['product_title'].fillna('').str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['query_len'], bins=30, color='#4C72B0', edgecolor='white')
axes[0].set_title('Query Length Distribution (words)', fontweight='bold')
axes[0].set_xlabel('Word Count')
axes[0].axvline(df['query_len'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["query_len"].mean():.1f}')
axes[0].legend()

axes[1].hist(df['title_len'].clip(0, 60), bins=30, color='#DD8452', edgecolor='white')
axes[1].set_title('Product Title Length Distribution (words)', fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].axvline(df['title_len'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["title_len"].mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/eda_text_lengths.png', bbox_inches='tight')
plt.show()

## 3. Query Length per Class

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df.boxplot(column='query_len', by='label_name', ax=ax,
           boxprops=dict(color='#4C72B0'),
           medianprops=dict(color='red', linewidth=2))
ax.set_title('Query Length by Class', fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Word Count')
plt.suptitle('')
plt.tight_layout()
plt.savefig('../outputs/figures/eda_query_len_by_class.png', bbox_inches='tight')
plt.show()

## 4. Top Query Terms

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
stop = set(stopwords.words('english'))

all_words = ' '.join(df['query'].dropna()).lower().split()
filtered = [w for w in all_words if w not in stop and len(w) > 2 and w.isalpha()]
top_words = Counter(filtered).most_common(20)

words, freqs = zip(*top_words)
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(list(reversed(words)), list(reversed(freqs)), color='#4C72B0')
ax.set_title('Top 20 Query Terms (excluding stopwords)', fontweight='bold')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.savefig('../outputs/figures/eda_top_query_terms.png', bbox_inches='tight')
plt.show()

## 5. Sample Examples per Class

In [ ]:
for label in ['E', 'S', 'C', 'I']:
    print(f"\n{'='*60}")
    print(f"Class: {label_map[label]}")
    print('='*60)
    sample = df[df['esci_label'] == label][['query', 'product_title']].sample(3, random_state=42)
    for _, row in sample.iterrows():
        print(f"  Query: {row['query']}")
        print(f"  Title: {row['product_title']}")
        print()